In [1]:
import pandas as pd
import numpy as np
import zipfile
import os
import joblib
from google.colab import files
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder

def run_train_and_save_pipeline():
    # 1. Upload the ZIP file
    print("--- STEP 1: Upload your .zip file ---")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded. Operation cancelled.")
        return

    zip_filename = list(uploaded.keys())[0]

    # 2. Extract
    extract_folder = 'data_extracted'
    if not os.path.exists(extract_folder):
        os.makedirs(extract_folder)

    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)
        all_files = zip_ref.namelist()

    # 3. Load CSV
    csv_files = [f for f in all_files if f.endswith('.csv') and not 'MACOSX' in f]
    if not csv_files:
        print("Error: No CSV file found inside the ZIP.")
        return

    csv_path = os.path.join(extract_folder, csv_files[0])
    df = pd.read_csv(csv_path)

    # 4. Feature Engineering & Encoding
    df['date'] = pd.to_datetime(df['date'])
    df['day_of_week'] = df['date'].dt.dayofweek

    # We save encoders in a dictionary to use them later
    encoders = {}
    cat_cols = ['product_id', 'category', 'vendor']

    for col in cat_cols:
        if col in df.columns:
            le = LabelEncoder()
            df[f'{col}_enc'] = le.fit_transform(df[col].astype(str))
            encoders[col] = le # Save the fitted encoder

    features = ['unit_price', 'quantity', 'total_price', 'day_of_week']
    for col in cat_cols:
        if f'{col}_enc' in df.columns:
            features.append(f'{col}_enc')

    X = df[features].fillna(0)

    # 5. Train Isolation Forest
    # Adjust contamination (0.05 = 5%) based on how many anomalies you expect
    model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
    df['anomaly_score'] = model.fit_predict(X)
    df['is_anomaly'] = df['anomaly_score'].apply(lambda x: 'Anomalous' if x == -1 else 'Normal')

    # 6. Show ALL Anomalies
    anomalies_only = df[df['is_anomaly'] == 'Anomalous']
    print("\n--- ALL DETECTED ANOMALIES ---")
    if not anomalies_only.empty:
        print(anomalies_only.to_string())
    else:
        print("No anomalies found.")

    # 7. Save Everything
    # Save the dataframe
    report_name = 'detailed_anomaly_report.csv'
    df.to_csv(report_name, index=False)

    # Save the model
    model_name = 'isolation_forest_model.pkl'
    joblib.dump(model, model_name)

    # Save the encoders (Required to process future data correctly)
    encoder_name = 'label_encoders.pkl'
    joblib.dump(encoders, encoder_name)

    print(f"\nTraining Complete. Downloading results and models...")

    # 8. Download files to your local machine
    for f in [report_name, model_name, encoder_name]:
        files.download(f)

# Run it
run_train_and_save_pipeline()

--- STEP 1: Upload your .zip file ---


Saving drive-download-20260327T142803Z-1-001.zip to drive-download-20260327T142803Z-1-001.zip

--- ALL DETECTED ANOMALIES ---
          date product_id      product_name     category  unit_price  quantity  total_price      vendor  day_of_week  product_id_enc  category_enc  vendor_enc  anomaly_score is_anomaly
20  2024-05-16       P001            Laptop  Electronics       56083         8       448664      Amazon            3               0             0           0             -1  Anomalous
56  2024-05-17       P001            Laptop  Electronics       56927         6       341562      Amazon            4               0             0           0             -1  Anomalous
69  2024-05-17       P001            Laptop  Electronics       51327        10       513270      Swiggy            4               0             0           3             -1  Anomalous
71  2024-05-17       P001            Laptop  Electronics       57974         9       524166  MakeMyTrip            4               0  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
import pandas as pd
import numpy as np
import zipfile
import os
import joblib
from google.colab import files
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder

# Set pandas to show all rows and full decimal precision in the console
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: '%.10f' % x)

def run_train_and_save_pipeline():
    # 1. Upload the ZIP file
    print("--- STEP 1: Upload your .zip file ---")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded. Operation cancelled.")
        return

    zip_filename = list(uploaded.keys())[0]

    # 2. Extract
    extract_folder = 'data_extracted'
    if not os.path.exists(extract_folder):
        os.makedirs(extract_folder)

    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)
        all_files = zip_ref.namelist()

    # 3. Load CSV
    csv_files = [f for f in all_files if f.endswith('.csv') and not 'MACOSX' in f]
    if not csv_files:
        print("Error: No CSV file found inside the ZIP.")
        return

    csv_path = os.path.join(extract_folder, csv_files[0])
    df = pd.read_csv(csv_path)

    # 4. Feature Engineering & Encoding
    df['date'] = pd.to_datetime(df['date'])
    df['day_of_week'] = df['date'].dt.dayofweek

    encoders = {}
    cat_cols = ['product_id', 'category', 'vendor']

    for col in cat_cols:
        if col in df.columns:
            le = LabelEncoder()
            df[f'{col}_enc'] = le.fit_transform(df[col].astype(str))
            encoders[col] = le

    features = ['unit_price', 'quantity', 'total_price', 'day_of_week']
    for col in cat_cols:
        if f'{col}_enc' in df_clean.columns if 'df_clean' in locals() else f'{col}_enc' in df.columns:
            features.append(f'{col}_enc')

    X = df[features].fillna(0)

    # 5. Train Isolation Forest
    model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
    model.fit(X)

    # Get the raw scores (High precision floats)
    # Lower scores = more anomalous
    df['raw_anomaly_score'] = model.decision_function(X)

    # Get the labels (-1 or 1)
    df['prediction_label'] = model.predict(X)
    df['is_anomaly'] = df['prediction_label'].apply(lambda x: 'Anomalous' if x == -1 else 'Normal')

    # 6. Show ALL Anomalies with full precision
    anomalies_only = df[df['is_anomaly'] == 'Anomalous'].sort_values(by='raw_anomaly_score')

    print("\n" + "="*50)
    print(f"DETECTION COMPLETE")
    print(f"Showing all {len(anomalies_only)} anomalies with raw decision scores:")
    print("="*50 + "\n")

    if not anomalies_only.empty:
        # to_string ensures everything is printed without truncation
        print(anomalies_only.to_string())
    else:
        print("No anomalies found.")

    # 7. Save Everything
    report_name = 'detailed_anomaly_report.csv'
    df.to_csv(report_name, index=False)

    model_name = 'isolation_forest_model.pkl'
    joblib.dump(model, model_name)

    encoder_name = 'label_encoders.pkl'
    joblib.dump(encoders, encoder_name)

    print(f"\nFiles ready. Downloading report and .pkl models...")

    # 8. Download
    for f in [report_name, model_name, encoder_name]:
        files.download(f)

# Run it
run_train_and_save_pipeline()

--- STEP 1: Upload your .zip file ---


Saving drive-download-20260327T142803Z-1-001.zip to drive-download-20260327T142803Z-1-001 (1).zip

DETECTION COMPLETE
Showing all 40 anomalies with raw decision scores:

          date product_id      product_name     category  unit_price  quantity  total_price      vendor  day_of_week  product_id_enc  category_enc  vendor_enc  raw_anomaly_score  prediction_label is_anomaly
570 2024-05-27       P001            Laptop  Electronics       46681        10       466810      Amazon            0               0             0           0      -0.0598049158                -1  Anomalous
604 2024-05-28       P001            Laptop  Electronics       54773        10       547730        Uber            1               0             0           4      -0.0556584603                -1  Anomalous
780 2024-05-31       P001            Laptop  Electronics       57791        10       577910        Uber            4               0             0           4      -0.0533678066                -1  Anomalous
67

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
import pandas as pd
import numpy as np
import zipfile

def generate_test_data():
    np.random.seed(42)

    # 1. Setup normal data patterns
    products = {
        'P001': {'name': 'Laptop', 'cat': 'Electronics', 'price': 55000},
        'P002': {'name': 'Printer', 'cat': 'Electronics', 'price': 15000},
        'P004': {'name': 'Desk', 'cat': 'Office', 'price': 9000},
        'P005': {'name': 'License', 'cat': 'Software', 'price': 20000}
    }
    vendors = ['Amazon', 'Flipkart', 'MakeMyTrip', 'Swiggy']

    data = []

    # Generate 80 NORMAL rows
    for _ in range(80):
        p_id = np.random.choice(list(products.keys()))
        row = {
            'date': '2024-04-01',
            'product_id': p_id,
            'product_name': products[p_id]['name'],
            'category': products[p_id]['cat'],
            'unit_price': products[p_id]['price'] + np.random.randint(-1000, 1000),
            'quantity': np.random.randint(1, 10),
            'vendor': np.random.choice(vendors)
        }
        row['total_price'] = row['unit_price'] * row['quantity']
        data.append(row)

    # Generate 20 ANOMALOUS rows (The "Bad" Data)
    for i in range(20):
        p_id = np.random.choice(list(products.keys()))

        # Mix of different anomaly types:
        if i % 3 == 0:
            # Type 1: Extreme Quantity (Buying 500 desks)
            u_price = products[p_id]['price']
            qty = 500
        elif i % 3 == 1:
            # Type 2: Wrong Price (A Laptop for 50 bucks)
            u_price = 50
            qty = np.random.randint(1, 5)
        else:
            # Type 3: Rare Vendor/Product Combo (Buying Flight Tickets from a Desk vendor)
            u_price = 150000 # Massive overcharge
            qty = 1

        row = {
            'date': '2024-04-01',
            'product_id': p_id,
            'product_name': products[p_id]['name'],
            'category': products[p_id]['cat'],
            'unit_price': u_price,
            'quantity': qty,
            'vendor': 'Unknown_Global_Store' # Unusual vendor
        }
        row['total_price'] = row['unit_price'] * row['quantity']
        data.append(row)

    # Shuffle and save
    df = pd.DataFrame(data).sample(frac=1).reset_index(drop=True)
    df.to_csv('test_procurement.csv', index=False)

    # Zip it up so you can test your upload pipeline
    with zipfile.ZipFile('test_data.zip', 'w') as zipf:
        zipf.write('test_procurement.csv')

    print("Success! 'test_data.zip' created with 100 rows (20 anomalies).")
    print("You can now download this from the sidebar and upload it to your main script.")

generate_test_data()

Success! 'test_data.zip' created with 100 rows (20 anomalies).
You can now download this from the sidebar and upload it to your main script.


```Testing```

In [5]:
import pandas as pd
import numpy as np
import joblib
from google.colab import files
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder

# --- FORMATTING SETTINGS ---
# Ensures full precision and no truncation in the display
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.15f' % x)

def run_csv_anomaly_pipeline():
    # 1. Upload the CSV file
    print("--- STEP 1: Upload your .csv file ---")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded. Operation cancelled.")
        return

    csv_filename = list(uploaded.keys())[0]

    # 2. Load Data
    df = pd.read_csv(csv_filename)
    print(f"Successfully loaded {len(df)} rows from {csv_filename}")

    # 3. Feature Engineering & Encoding
    # Ensure date is handled
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        df['day_of_week'] = df['date'].dt.dayofweek
    else:
        df['day_of_week'] = 0 # Fallback if date is missing

    encoders = {}
    cat_cols = ['product_id', 'category', 'vendor']

    # Encode categorical text into numbers for the model
    for col in cat_cols:
        if col in df.columns:
            le = LabelEncoder()
            df[f'{col}_enc'] = le.fit_transform(df[col].astype(str))
            encoders[col] = le

    # Define the mathematical features the model will analyze
    features = ['unit_price', 'quantity', 'total_price', 'day_of_week']
    for col in cat_cols:
        if f'{col}_enc' in df.columns:
            features.append(f'{col}_enc')

    X = df[features].fillna(0)

    # 4. Train Isolation Forest
    # contamination=0.2 means we expect 20% anomalies (matching your 20/100 test case)
    model = IsolationForest(n_estimators=100, contamination=0.2, random_state=42)
    model.fit(X)

    # Calculate scores (The more negative, the more anomalous)
    df['raw_score'] = model.decision_function(X)
    df['prediction_label'] = model.predict(X)
    df['is_anomaly'] = df['prediction_label'].apply(lambda x: 'ANOMALY' if x == -1 else 'Normal')

    # 5. Show ALL Results (Sorted by most anomalous first)
    anomalies_only = df[df['is_anomaly'] == 'ANOMALY'].sort_values(by='raw_score')

    print("\n" + "="*90)
    print(f"ANOMALY DETECTION RESULTS (Found {len(anomalies_only)} suspicious rows)")
    print("="*90)

    if not anomalies_only.empty:
        # Display the core columns + the precise raw score
        display_cols = ['product_name', 'unit_price', 'quantity', 'total_price', 'vendor', 'raw_score', 'is_anomaly']
        print(anomalies_only[display_cols].to_string(index=False))
    else:
        print("No anomalies detected.")

    # 6. Save & Download Files
    report_name = 'anomaly_analysis_report.csv'
    model_name = 'isolation_forest_model.pkl'
    encoder_name = 'label_encoders.pkl'

    df.to_csv(report_name, index=False)
    joblib.dump(model, model_name)
    joblib.dump(encoders, encoder_name)

    print(f"\nProcessing complete. Downloading report and .pkl models...")
    for f in [report_name, model_name, encoder_name]:
        files.download(f)

# Run the pipeline
run_csv_anomaly_pipeline()

--- STEP 1: Upload your .csv file ---


Saving test_procurement.csv to test_procurement (2).csv
Successfully loaded 100 rows from test_procurement (2).csv

ANOMALY DETECTION RESULTS (Found 20 suspicious rows)
product_name  unit_price  quantity  total_price               vendor          raw_score is_anomaly
      Laptop       55000       500     27500000 Unknown_Global_Store -0.229822760779667    ANOMALY
      Laptop      150000         1       150000 Unknown_Global_Store -0.137605098575453    ANOMALY
     License       20000       500     10000000 Unknown_Global_Store -0.108944554078734    ANOMALY
     License       20000       500     10000000 Unknown_Global_Store -0.108944554078734    ANOMALY
     License       20000       500     10000000 Unknown_Global_Store -0.108944554078734    ANOMALY
     License       20000       500     10000000 Unknown_Global_Store -0.108944554078734    ANOMALY
     Printer       15000       500      7500000 Unknown_Global_Store -0.103393785883853    ANOMALY
     Printer       15000       500     

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>